# Demo: Klassifisering av kredittkortklager

Denne notebooken er en kort, kjørbar versjon av prosjektets sentrale metode, nemlig å sammenligne hvor godt en logistisk regresjonsmodell og en stor språkmodell klassifiserer kredittkortklager. Demoen baserer seg på et lite utvalg av dataene fra hovedanalysen.

**Oppsett:**
1. Trener den balanserte logistiske regresjonsmodellen (med TF-IDF-vektorer) på treningssettet. Bruker samme oppsett som i notebook 1.
2. Trekker et stratifisert utvalg fra testsettet (350 klager, med samme fordeling mellom kategoriene som i testsettet, fast startverdi), og klassifiserer det med baselinen, den logistiske regresjonsmodellen og LLM-en. For språkmodellen er modellen, prompten og innstillingene de samme som i notebook 2).
3. Sammenligner LLM-svarene med svarene fra kjøringen 23.10.2026 (lagret i `llm_test.csv`) og viser hvor mange klager som fikk et annet svar. Det illustrerer at svarene kan variere fra kjøring til kjøring med temperatur lik 1.
4. Viser accuracy, macro-F1, F1 per kategori og noen eksempler der modellene er uenige.

   
**Data:** 
- Demoen leser `treningssett.csv`, `testsett.csv` og `llm_test.csv`, som ligger i repoet.
- Notebook 1–3 trenger ikke kjøres først.
- Rådatafilen `Kundeklager.csv` er ikke med i repoet fordi den er for stor, men den trengs bare for å kjøre notebook 1 og ikke for demoen.

**Kjøretid:** Omtrent to minutter.

**API-kostnad:** Omtrent 0,03 USD. Kostnaden anslås før API-kallene, og demoen stopper hvis anslaget overstiger 1 USD.

**API-nøkkel:** Demoen krever en OpenAI API-nøkkel i en `.env`-fil i prosjektmappen (se README.md). Mangler nøkkelen, stopper demoen med en feilmelding før noe annet kjøres. Alle API-kall logges i `logs/demo_logg.jsonl`.

In [16]:
# Importerer nødvendige libraries
import os
import json
import time
import textwrap
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [17]:
# Viser hele kategorinavnene i tabellene
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

# Starter klokken for å måle hvor lang tid hele demoen tar
demo_start = time.time()

### Innstillinger

In [31]:
# Antall klager som trekkes fra testsettet (omtrent 10 % av de 3 583 klagene)
ANTALL_KLAGER = 350

# Fast startverdi, slik at de samme klagene trekkes hver gang demoen kjøres
STARTVERDI = 1

# Samme modell som i notebook 2
MODELL = 'gpt-6-luna'

# Pris i USD per million tokens for GPT-6 Luna 
PRIS_INPUT = 0.10
PRIS_OUTPUT = 0.50

# Demoen stopper før API-kallene hvis den anslåtte kostnaden er høyere enn 1 USD
MAKS_KOSTNAD = 1.00

# Lager en mappe for loggen hvis den ikke finnes
os.makedirs('logs', exist_ok=True)
LOGGFIL = os.path.join('logs', 'demo_logg.jsonl')

# Leser API-nøkkelen fra .env-filen og stopper demoen hvis den mangler
load_dotenv()
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Fant ingen API-nøkkel. Lag en fil som heter .env i prosjektmappen '
                       'med linjen OPENAI_API_KEY=din-nøkkel-her (se README.md).')

### Leser inn data

In [19]:
# Leser inn treningssettet og testsettet fra notebook 1, og LLM-svarene fra notebook 2
train_df = pd.read_csv('treningssett.csv')
test_df = pd.read_csv('testsett.csv')
llm_test_df = pd.read_csv('llm_test.csv')

# Trekker 350 klager fra testsettet med samme fordeling mellom kategoriene som i testsettet
utvalg_df, _ = train_test_split(test_df, train_size=ANTALL_KLAGER,
                                stratify=test_df['Issue'], random_state=STARTVERDI)
utvalg_df = utvalg_df.reset_index(drop=True)

# Legger til svarene LLM-en ga på disse klagene i kjøringen 23.10.2026
lagrede_svar = llm_test_df[['Complaint ID', 'svar_llm']].rename(columns={'svar_llm': 'svar_llm_lagret'})
utvalg_df = utvalg_df.merge(lagrede_svar, on='Complaint ID', how='left')

print('Treningssett:', len(train_df), 'klager')
print('Utvalg fra testsettet:', len(utvalg_df), 'klager')
print(utvalg_df['Issue'].value_counts())

Treningssett: 12537 klager
Utvalg fra testsettet: 350 klager
Issue
Problem with a purchase shown on your statement            90
Other features, terms, or problems                         60
Fees or interest                                           58
Problem when making payments                               42
Closing your account                                       34
Getting a credit card                                      33
Advertising and marketing, including promotional offers    33
Name: count, dtype: int64


### Trener den balanserte logistiske regresjonsmodellen

In [20]:
start = time.time()

# Gjør klagetekstene om til TF-IDF-vektorer (ordforrådet læres kun fra treningssettet)
tf_idf_vektor = TfidfVectorizer()
train_X_vek = tf_idf_vektor.fit_transform(train_df['Consumer complaint narrative'])

# Trener den balanserte logistiske regresjonsmodellen
reg = LogisticRegression(max_iter=1000, class_weight='balanced')
reg.fit(train_X_vek, train_df['Issue'])

print(f'Modellen er trent på {time.time() - start:.0f} sekunder')

Modellen er trent på 8 sekunder


### Logistisk regresjon på utvalget

In [21]:
# Gjør utvalget om til TF-IDF-vektorer med samme ordforråd som treningssettet
utvalg_X_vek = tf_idf_vektor.transform(utvalg_df['Consumer complaint narrative'])

# Lar den logistiske regresjonsmodellen predikere kategori for hver klage i utvalget
utvalg_df['lr_pred'] = reg.predict(utvalg_X_vek)

print('Logistisk regresjon har klassifisert', len(utvalg_df), 'klager')

Logistisk regresjon har klassifisert 350 klager


### Baseline 

In [22]:
# Finner den største kategorien i treningssettet og bruker den som svar for alle klagene
største_kategori = train_df['Issue'].value_counts().idxmax()
utvalg_df['baseline'] = største_kategori

print('Baseline gjetter alltid:', største_kategori)

Baseline gjetter alltid: Problem with a purchase shown on your statement


### LLM med prompt B

In [23]:
# Problemkategoriene i alfabetisk rekkefølge (samme rekkefølge som i notebook 2)
kategorier = sorted(train_df['Issue'].unique())

# Kort beskrivelse av hver kategori, basert på underkategoriene i datasettet
beskrivelser = {
    'Problem with a purchase shown on your statement': 'disputes about purchases not being resolved, charges for something not purchased, or being overcharged',
    'Fees or interest': 'problems with fees, too much interest charged, or unexpected increases in interest rate',
    'Problem when making payments': 'problems during the payment process, or not receiving a bill or not knowing a payment was due',
    'Closing your account': 'the company closed the account, or the consumer cannot close the account',
    'Getting a credit card': 'cards opened through identity theft or fraud, denied applications, cards never applied for, delays in applications, or problems getting a replacement card',
    'Advertising and marketing, including promotional offers': 'not receiving advertised or promotional terms, or confusing or misleading advertising',
    'Other features, terms, or problems': 'rewards, customer service, balance transfers, privacy, add-on products, cash advances, arbitration, convenience checks, or other problems',
}

# Lager prompt B: kategorinavn med beskrivelser og regler
def prompt_b(klage):
    kategoriliste = '\n'.join(f'- {k}: {beskrivelser[k]}' for k in kategorier)
    return f'''Classify the following credit card complaint into exactly one of these categories:
{kategoriliste}

If the complaint fits several categories, choose the one that describes the main problem.
Use "Other features, terms, or problems" only if none of the other categories fit.

XXXX indicates anonymized information.
Answer with the category name only, exactly as written before the colon.

Complaint:
{klage}'''

#### Kostnadsanslag før API-kallene

In [24]:
# Anslår antall tokens og kostnad for hele utvalget
anslått_input = sum(len(prompt_b(k)) for k in utvalg_df['Consumer complaint narrative']) / 4
anslått_output = 20 * len(utvalg_df)
anslått_kostnad = (anslått_input * PRIS_INPUT + anslått_output * PRIS_OUTPUT) / 1_000_000

print(f'Anslått antall input-tokens: {anslått_input:,.0f}')
print(f'Anslått kostnad: {anslått_kostnad:.4f} USD')

# Stopper demoen hvis kostnaden blir for høy
assert anslått_kostnad < MAKS_KOSTNAD, 'Anslått kostnad er over grensen – reduser ANTALL_KLAGER'

Anslått antall input-tokens: 236,958
Anslått kostnad: 0.0272 USD


#### Klassifiserer utvalget

In [25]:
# Kobler til OpenAI med API-nøkkelen fra .env
client = OpenAI()

# Klassifiserer én klage og returnerer svar og tokenbruk
def klassifiser_en(klage):
    # Prøver opptil tre ganger i tilfelle API-et feiler
    for forsøk in range(3):
        try:
            svar = client.responses.create(
                model=MODELL,
                reasoning={'effort': 'none'},
                input=prompt_b(klage)
            )
            tekst = svar.output_text.strip().rstrip('.')
            return tekst, svar.usage.input_tokens, svar.usage.output_tokens
        except Exception as feilmelding:
            print('Feil –', feilmelding)
            time.sleep(5)
    return None, 0, 0

# Sender åtte klager samtidig
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    resultater = list(pool.map(klassifiser_en, utvalg_df['Consumer complaint narrative']))

utvalg_df['svar_llm'] = [r[0] for r in resultater]
brukte_input = sum(r[1] for r in resultater)
brukte_output = sum(r[2] for r in resultater)

# Logger hvert kall som én linje i loggfilen
tidspunkt = datetime.now().isoformat(timespec='seconds')
with open(LOGGFIL, 'a', encoding='utf-8') as fil:
    for (_, rad), (svar, inn, ut) in zip(utvalg_df.iterrows(), resultater):
        fil.write(json.dumps({
            'tidspunkt': tidspunkt,
            'complaint_id': int(rad['Complaint ID']),
            'modell': MODELL,
            'innstillinger': {'reasoning_effort': 'none', 'temperatur': 'standard (1)'},
            'prompt': 'B',
            'fasit': rad['Issue'],
            'svar': svar,
            'gyldig_svar': svar in kategorier,
            'input_tokens': inn,
            'output_tokens': ut,
        }, ensure_ascii=False) + '\n')

print(f'{len(utvalg_df)} klager klassifisert med {MODELL} på {time.time() - start:.0f} sekunder')
print('Loggen er lagret i', LOGGFIL)

# Sammenligner med svarene fra kjøringen 23.10.2026 (temperatur 1 kan gi ulike svar)
ulike = (utvalg_df['svar_llm'] != utvalg_df['svar_llm_lagret']).sum()
print(f'{ulike} av {len(utvalg_df)} klager fikk et annet svar enn i kjøringen 23.10.2026')

350 klager klassifisert med gpt-6-luna på 62 sekunder
Loggen er lagret i logs\demo_logg.jsonl
30 av 350 klager fikk et annet svar enn i kjøringen 23.10.2026


### Kontroll av svarene

In [26]:
# Sjekker at alle klager fikk et svar, og at svaret er en av de sju kategoriene
print('Manglende svar:', utvalg_df['svar_llm'].isna().sum())
print('Ugyldige svar:', (~utvalg_df['svar_llm'].isin(kategorier)).sum())

Manglende svar: 0
Ugyldige svar: 0


### Resultater på utvalget

In [27]:
# Beregner accuracy og macro-F1 for hver metode
metoder = {'Baseline': 'baseline', 'Logistisk regresjon': 'lr_pred', 'LLM (prompt B)': 'svar_llm'}

resultat_df = pd.DataFrame({
    navn: {
        'Accuracy': accuracy_score(utvalg_df['Issue'], utvalg_df[kolonne]),
        'Macro-F1': f1_score(utvalg_df['Issue'], utvalg_df[kolonne], average='macro', zero_division=0),
    }
    for navn, kolonne in metoder.items()
}).T.round(2)

print(resultat_df)

                     Accuracy  Macro-F1
Baseline                 0.26      0.06
Logistisk regresjon      0.71      0.69
LLM (prompt B)           0.70      0.69


In [28]:
# F1 per kategori for de to modellene
f1_df = pd.DataFrame({
    'Logistisk regresjon': f1_score(utvalg_df['Issue'], utvalg_df['lr_pred'], labels=kategorier, average=None, zero_division=0),
    'LLM (prompt B)': f1_score(utvalg_df['Issue'], utvalg_df['svar_llm'], labels=kategorier, average=None, zero_division=0),
}, index=kategorier).round(2)

print(f1_df)

                                                         Logistisk regresjon  LLM (prompt B)
Advertising and marketing, including promotional offers                 0.56            0.63
Closing your account                                                    0.62            0.75
Fees or interest                                                        0.69            0.62
Getting a credit card                                                   0.77            0.79
Other features, terms, or problems                                      0.56            0.58
Problem when making payments                                            0.78            0.62
Problem with a purchase shown on your statement                         0.85            0.86


### Eksempler der modellene er uenige

In [29]:
# Velger tre klager der LR og LLM-en valgte ulik kategori
uenige_df = utvalg_df[utvalg_df['lr_pred'] != utvalg_df['svar_llm']]
print(len(uenige_df), 'av', len(utvalg_df), 'klager fikk ulik kategori fra de to modellene\n')

for _, rad in uenige_df.head(3).iterrows():
    print('Fasit (kundens valg):', rad['Issue'])
    print('Logistisk regresjon: ', rad['lr_pred'])
    print('LLM (prompt B):      ', rad['svar_llm'])
    print(textwrap.fill(rad['Consumer complaint narrative'][:600] + ' ...', width=100))
    print('_' * 100)

104 av 350 klager fikk ulik kategori fra de to modellene

Fasit (kundens valg): Fees or interest
Logistisk regresjon:  Advertising and marketing, including promotional offers
LLM (prompt B):       Getting a credit card
On XX/XX/XXXX, I signed an agreement for a loan with Iron Capital Group for {$30000.00}. My son
spoke with Iron Capital on XX/XX/XXXX before filling out the agreement and ask if this was a credit
card situation because he doesn't want it which they advised that it wasn't. They advised that they
had lenders who were offering loans and you just got ta pay 15 % of the amount you requested. I
tried printing the agreement when it was sent to us to take home to read before filling out but I
couldn't because it will not allow you to and it gives you no option. They advise me that I needed
to fill it  ...
____________________________________________________________________________________________________
Fasit (kundens valg): Problem when making payments
Logistisk regresjon:  Pr

### Oppsummering av kjøringen

In [30]:
faktisk_kostnad = (brukte_input * PRIS_INPUT + brukte_output * PRIS_OUTPUT) / 1_000_000

print('Modell:', MODELL, '| reasoning effort = none | temperatur = standard (1)')
print('Antall klager:', len(utvalg_df))
print(f'Tokens brukt: {brukte_input:,} input, {brukte_output:,} output')
print(f'Faktisk kostnad: {faktisk_kostnad:.4f} USD')
print(f'Total kjøretid: {time.time() - demo_start:.0f} sekunder')

Modell: gpt-6-luna | reasoning effort = none | temperatur = standard (1)
Antall klager: 350
Tokens brukt: 193,108 input, 3,339 output
Faktisk kostnad: 0.0210 USD
Total kjøretid: 71 sekunder
